# GymRec: train the two-stage exercise classifier on Google Colab

<a href="https://colab.research.google.com/github/Sarthak-D96/Gym_Exercise/blob/main/training/colab_two_stage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset: [Gym Exercise DataSet (68K frames, 22 classes)](https://www.kaggle.com/datasets/amanyadav2030a/processed-train-test-exercise-68k)
by Aman Yadav (CC0). Each video is a folder of JPEG frames at about 5 fps.

**Two-stage model**

```
16-frame window ─► router: which GROUP? (legs / press / pull / arms / core)
                     └─► specialist of that group: which EXERCISE? (e.g. deadlift vs romanian deadlift)
```

* Specialists get 13 extra geometry features (torso tilt, shin angle, wrist height, grip width, …) aimed at look-alike exercises.
* Every stage tries RF / XGBoost / LSTM / GRU / CNN and keeps its best one **on validation**, so stages can use different model types.
* The whole chain is then scored on the test set, next to the single model if you trained one before (same `WORK_DIR`).
* Output: `gymrec_model_two_stage-<date>.zip`, installed on the website with the same `install_model` command.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. **Kaggle API key:** kaggle.com → Settings → API → *Create New Token* (downloads `kaggle.json`).
   In Colab open the 🔑 **Secrets** panel (left sidebar), add `KAGGLE_USERNAME` and `KAGGLE_KEY` from that file,
   and switch on notebook access for both.
3. **Runtime → Run all.** Everything is saved to Google Drive (`MyDrive/gymrec`). If Colab disconnects, reconnect
   and Run all again: extracted keypoints, the built dataset and finished models are reused, so it resumes.

In [ ]:
# 1. GPU check + Google Drive (all outputs are kept there)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")
WORK_DIR = "/content/drive/MyDrive/gymrec/work"   # same folder as the single-model notebook: keypoints are reused

In [ ]:
# 2. Get the code from GitHub
%cd /content
!test -d Gym_Exercise || git clone -q https://github.com/Sarthak-D96/Gym_Exercise.git
%cd /content/Gym_Exercise
!git pull -q
!pip install -q ultralytics kagglehub xgboost
!test -f training/select_two_stage.py || echo "!! This copy of the repo has no two-stage code: push it to GitHub first."
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 3. Download the dataset from Kaggle (~19 GB; 5–15 minutes, cached for this session)
import os
from google.colab import userdata

for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        raise SystemExit(f"Add the Colab secret {k} (see the top of the notebook) and allow notebook access.")

import kagglehub
DATASET_ROOT = kagglehub.dataset_download("amanyadav2030a/processed-train-test-exercise-68k")
print("Downloaded to", DATASET_ROOT)

In [ ]:
# 4. Find the train / test folders (one sub-folder per exercise)
from pathlib import Path

def find_dir(name):
    hits = sorted(p for p in Path(DATASET_ROOT).rglob(name) if p.is_dir())
    return max(hits, key=lambda p: len(p.parts)) if hits else None   # deepest if nested

TRAIN_DIR, TEST_DIR = find_dir("processed-train"), find_dir("processed-test")
print("train:", TRAIN_DIR)
print("test: ", TEST_DIR)
if TRAIN_DIR is None:
    raise SystemExit("processed-train not found: look at the folders under DATASET_ROOT and set TRAIN_DIR by hand.")

folders = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
print(len(folders), "exercise folders:", folders)
print("example video folders:", sorted(p.name for p in (TRAIN_DIR / folders[0]).iterdir())[:12])

## 5. Configure

Defaults come from `training/gym_config.py`; anything set here overrides them for this run.

* **`CLASSES`**: the exercises to recognise (slugs or the dataset's folder names). The website shows exactly these after install.
* **`GROUPS`**: which exercises share a specialist. Put look-alikes in the same group. Classes in no group get their own;
  a group with one exercise needs no specialist.
* **`MODELS`**: model types every stage tries. Fewer types train faster (e.g. `["xgb", "cnn"]`).
* **`GROUP_REGEX`**: if the video folder names above show clips cut from one source video (e.g. `vid-squat_12_1`,
  `vid-squat_12_2`), set a regex whose group 1 is the source id. Then clips of one video never sit in both
  train and validation, and the validation scores are honest.

In [ ]:
import sys; sys.path.insert(0, "/content/Gym_Exercise/training")
import run_all
config = run_all.config   # training/gym_config.py; values set below override it

# all 22 dataset classes (lunge and jumping_jack are not in this dataset)
config.CLASSES = [c for c in config.EXERCISES if c not in ("lunge", "jumping_jack")]

config.TWO_STAGE = True
config.GROUPS = {
    "legs":  ["squat", "deadlift", "romanian_deadlift", "leg_extension", "hip_thrust"],
    "press": ["push_up", "bench_press", "incline_bench_press", "decline_bench_press", "shoulder_press",
              "chest_fly_machine", "tricep_dips"],
    "pull":  ["pull_up", "lat_pulldown", "t_bar_row"],
    "arms":  ["bicep_curl", "hammer_curl", "tricep_pushdown", "lateral_raise"],
    "core":  ["plank", "leg_raises", "russian_twist"],
}

config.MODELS = ["rf", "xgb", "lstm", "gru", "cnn"]
config.SELECT_BY = "macro_f1"

# Accuracy improvements (see training/README.md, "Improving the score")
config.GROUP_BY = "auto"        # keep clips of one source video in one split -> honest validation
config.AUGMENT = "strong"       # camera yaw/scale, corrupted keypoints, dropped frames (deep models)
config.AUGMENT_COPIES = 1       # augmented copies of the training data for RF / XGBoost
config.ENSEMBLE_MAX = 2         # per stage, average the top-2 model types if validation improves
# config.GROUP_REGEX = r"^(.*)_\d+$"     # if the folder names show the source video (overrides GROUP_BY)
# config.RETRAIN = True                   # retrain finished models (e.g. after changing AUGMENT or DL_PARAMS)

config.DATA_DIRS = [str(TRAIN_DIR)]
config.TEST_DIRS = [str(TEST_DIR)] if TEST_DIR else []
config.WORK_DIR = WORK_DIR
config.FRAMES_FPS = 5    # the dataset is frames at ~5 fps: use every frame
config.FPS = 5

In [ ]:
# 6. Check: dataset folder -> class, and the group plan (router + specialists)
counts = run_all.preview()

In [ ]:
# 7. Run everything: keypoints -> dataset -> train every stage -> pick the best per stage -> test -> export
#    (a few hours for 22 classes and 5 model types; Run all again after a disconnect to resume)
run_all.main()

In [ ]:
# 8. Results and download
from pathlib import Path
from google.colab import files

latest = sorted(Path(WORK_DIR).glob("gymrec_model_two_stage-*.zip"), key=lambda p: p.stat().st_mtime)[-1]
print(open(latest.with_suffix("") / "results.md").read())
print("Model zip:", latest, "(also kept in Google Drive)")
files.download(str(latest))

In [ ]:
# Optional: also train the single model on the same (now honestly split) data, to compare both in score_history
# config.TWO_STAGE = False
# run_all.main(["train", "select"])
# config.TWO_STAGE = True


## 9. Connect it to the website (on your laptop)

```bash
cd ~/Desktop/Gym_Exercise
source .venv/bin/activate
pip install -r requirements.txt            # if a stage chose XGBoost: pip install xgboost
python manage.py install_model ~/Downloads/gymrec_model_two_stage-<date>.zip
python manage.py runserver
```

`install_model` unpacks the zip into `ml_models/`, lists the stages, adds missing exercises,
activates the two-stage classifier and the pose weights, and shows only the trained exercises.
To switch back to an earlier model, run `install_model` with that model's zip.

For the journal: copy `results.md` (printed above) into Phase 4. It compares every stage and the whole chain
with the single model.